# 00 · Primeiro contato com os dados — Rastreamento de HbA1c elevada (PNS 2013)

**Disciplina:** Machine Learning · IFMS Campus Três Lagoas · Projeto Final
**Integrantes:** Gabriela Tucunduva, Luisa de Matos e Maria Luisa Latta
**Etapa:** Entrega 1 — problema, dados, métrica e baseline

## Ficha da proposta

| Campo | Preenchimento |
|---|---|
| **Problema** | Dado o perfil (idade, sexo, peso, altura, hipertensão e hábitos de alimentação e exercício) de um adulto sem diagnóstico prévio de diabetes, prever se a hemoglobina glicada (HbA1c) está elevada (≥ 6,5%), para que a pessoa possa ser orientada a procurar avaliação e confirmação por um profissional de saúde. |
| **Usuário** | Um adulto que preenche o formulário. Com a resposta, decide se procura avaliação e exame de confirmação. |
| **Tipo e alvo** | Classificação binária. Alvo: `hba1c_alterada` (1 = HbA1c ≥ 6,5%; 0 = HbA1c < 6,5%), construído a partir da coluna `Z034`. |
| **Entradas** | 10 campos: idade, sexo, peso, altura, hipertensão, dias por semana de feijão, frutas, refrigerante e doces, e exercício. Todos passam no teste do formulário: uma pessoa consegue informá-los sem fazer exame. |
| **Dataset** | PNS 2013, exames laboratoriais: 8.952 linhas × 509 colunas (detalhes na seção 2). |
| **Métrica e baseline** | Recall da classe positiva. O baseline `Dummy` ("sempre não") tem recall 0. |
| **Riscos** | Rótulo de rastreamento (não é diagnóstico), classe positiva rara, peso e altura com ausentes, colunas de saúde e sexo. Detalhes na seção 7. |

> **Limite clínico essencial:** uma única medida de HbA1c ≥ 6,5% não confirma, sozinha, diagnóstico de diabetes. Neste projeto ela é a referência laboratorial de um exercício acadêmico de rastreamento. A saída não é diagnóstico e exige avaliação profissional.

## 1. Bibliotecas e dados

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, precision_score, f1_score

SEED = 42

# Os arquivos são lidos direto do repositório do grupo no GitHub
URL_EXAMES = "https://github.com/tucunduvas/diabetes_risk_prediction/raw/refs/heads/main/data/exames-PNS-2013-final.xlsx"
df = pd.read_excel(URL_EXAMES)

print(f"Dimensões da planilha: {df.shape[0]:,} linhas × {df.shape[1]:,} colunas")

Dimensões da planilha: 8,952 linhas × 509 colunas


## 2. Dataset: fonte, licença e dicionário

- **Fonte:** Pesquisa Nacional de Saúde (PNS) 2013, base dos exames laboratoriais, realizada pelo IBGE em parceria com o Ministério da Saúde e a Fiocruz. A planilha tem 8.952 participantes, uma amostra com dados de exames, e não toda a amostra entrevistada.
- **Link (microdados):** https://www.ibge.gov.br/estatisticas/sociais/saude/9160-pesquisa-nacional-de-saude.html?=&t=microdados
- **Nota técnica dos exames:** https://ftp.ibge.gov.br/PNS/2013/Divulgacoes/Outros/Exames/Nota_tecnica_Exames_Laboratoriais_PNS.pdf
- **Licença:** não identificamos uma licença formal (como Creative Commons) para esta base. O artigo metodológico dos exames (Szwarcwald et al., *Rev Bras Epidemiol* 2019; 22 Supl 2: e190004) informa que ela foi disponibilizada sem necessidade de autorização prévia para uso. O uso aqui é acadêmico, sempre citando a fonte.
- **Arquivos no repositório (`data/`):** `exames-PNS-2013-final.xlsx` e `dicionario-variaveis-exames-pns-2013.xlsx`.

**Dicionário das colunas de entrada** (códigos conferidos no dicionário da PNS):

| Campo do formulário | Coluna | Tipo | Códigos ou unidade |
|---|---|---|---|
| Idade | `Z002` | numérica | anos |
| Sexo | `Z001` | categórica | 1 = Masculino, 2 = Feminino |
| Peso aproximado | `P00101` | numérica | kg (autorreferido) |
| Altura aproximada | `P00401` | numérica | cm (autorreferida) |
| Hipertensão diagnosticada | `Q002` | categórica | 1 = Sim, 2 = Apenas durante a gravidez, 3 = Não |
| Dias por semana com feijão | `P006` | numérica | 0 a 7 dias |
| Dias por semana com frutas | `P018` | numérica | 0 a 7 dias |
| Dias por semana com refrigerante ou suco artificial | `P020` | numérica | 0 a 7 dias |
| Dias por semana com doces | `P025` | numérica | 0 a 7 dias |
| Exercício nos últimos 3 meses | `P034` | categórica | 1 = Sim, 2 = Não |

**Colunas auxiliares, que não entram como entrada do modelo:**

- `Q030`: diagnóstico médico prévio de diabetes (1 = Sim, 2 = Apenas durante a gravidez, 3 = Não). Só define a população do estudo.
- `P005`: gravidez atual (1 = Sim, 2 = Não, 3 = Não sabe). Só define a população do estudo.
- `Z034`: HbA1c em %. Só constrói o alvo. No dicionário, `.` significa "amostra inadequada".

Os números 1, 2 e 3 das categóricas são códigos, não quantidades: na E2 elas são tratadas como categorias.

## 3. Recorte da população e definição do alvo

O objetivo do grupo não é estimar se a pessoa já recebeu um diagnóstico, e sim estudar um possível sinal laboratorial em pessoas que **não relataram diagnóstico prévio**. Critérios:

1. `Q030 == 3`: respondeu que **não** recebeu diagnóstico médico de diabetes.
2. `Z034` preenchida: existe resultado de HbA1c.
3. `P005 != 1`: exclui quem informou estar grávida, pois a gestação exige interpretação clínica específica.

O alvo é criado só a partir de `Z034`: `1` quando HbA1c ≥ 6,5% e `0` quando HbA1c < 6,5%.

In [2]:
base = df[(df["Q030"] == 3) & (df["P005"] != 1)].dropna(subset=["Z034"]).copy()
base["hba1c_alterada"] = (base["Z034"] >= 6.5).astype(int)

print("Participantes na planilha:", len(df))
print("Participantes após os critérios:", len(base))

contagem = base["hba1c_alterada"].value_counts()
display(pd.DataFrame({
    "quantidade": contagem,
    "percentual (%)": (base["hba1c_alterada"].value_counts(normalize=True) * 100).round(2),
}))

Participantes na planilha: 8952
Participantes após os critérios: 6832


,quantidade,percentual (%)
hba1c_alterada,,
0,6573,96.21
1,259,3.79


**Observação:** o alvo é muito desbalanceado: 6.573 participantes (96,21%) com HbA1c < 6,5% e 259 (3,79%) com HbA1c ≥ 6,5%.

## 4. Entradas, valores ausentes e divisão treino/teste

In [3]:
FEATURES = ["Z002", "Z001", "P00101", "P00401", "Q002", "P006", "P018", "P020", "P025", "P034"]

X = base[FEATURES]
y = base["hba1c_alterada"]

print("X:", X.shape, "| y:", y.shape)
display((X.isna().mean() * 100).round(2).sort_values(ascending=False).to_frame("% ausente"))

X: (6832, 10) | y: (6832,)


,% ausente
P00401,26.86
P00101,14.45
Q002,1.54
Z002,0.00
Z001,0.00
P006,0.00
P018,0.00
P020,0.00
P025,0.00
P034,0.00


**Observação:** peso e altura autorreferidos têm ausentes, porque nem todos os entrevistados sabiam informar esses dados. Não vamos excluir essas linhas: o tratamento será feito dentro do pipeline, na Entrega 2.

**Divisão treino/teste:** 80% para treino e validação cruzada e 20% para o teste final, que fica trancado até a Entrega 3. `stratify=y` mantém a proporção das classes e `random_state=42` torna a divisão reproduzível.

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

print("Treino:", X_tr.shape, "| Teste (trancado):", X_te.shape)
print(f"Classe positiva no treino: {y_tr.mean():.2%}")

Treino: (5465, 10) | Teste (trancado): (1367, 10)
Classe positiva no treino: 3.79%


## 5. Métrica e baseline

**Métrica principal: recall da classe positiva.** O projeto é de rastreamento: deixar de sinalizar alguém cuja HbA1c está elevada (falso negativo) é o erro mais caro, e um alarme falso custa menos, pois leva a um exame de confirmação. Também acompanhamos:

- **precisão:** quantos alertas correspondem a casos positivos, para não gerar falsos positivos demais;
- **F1 da classe positiva** e **F1-macro:** equilibram recall e precisão (o F1-macro considera as duas classes);
- **acurácia balanceada:** média do recall das duas classes;
- **acurácia:** só como informação complementar, pois fica alta mesmo quando o modelo não acerta nenhum positivo.

O baseline é o `DummyClassifier` que sempre prevê a classe mais frequente. Ele é medido por validação cruzada estratificada de 5 dobras, somente no treino. Essas mesmas métricas e essa mesma validação serão usadas na E2 e na E3.

In [5]:
baseline = DummyClassifier(strategy="most_frequent")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scoring = {
    "acuracia": "accuracy",
    "recall_positivo": "recall",
    "precisao_positivo": make_scorer(precision_score, zero_division=0),
    "f1_positivo": make_scorer(f1_score, zero_division=0),
    "f1_macro": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
}

resultados = cross_validate(baseline, X_tr, y_tr, cv=cv, scoring=scoring)

media = pd.DataFrame(resultados).filter(like="test_").mean()
media.index = media.index.str.replace("test_", "")
display(media.round(4).to_frame("Dummy (média da validação cruzada)"))

,Dummy (média da validação cruzada)
acuracia,0.9621
recall_positivo,0.0000
precisao_positivo,0.0000
f1_positivo,0.0000
f1_macro,0.4903
balanced_accuracy,0.5000


**Observação:** o baseline tem acurácia de 96,21% e, mesmo assim, recall, precisão e F1 da classe positiva iguais a **0**, porque responde sempre "não". O F1-macro é 0,4903 e a acurácia balanceada é 0,50.

**Decisão:** acurácia não serve para avaliar este projeto. Na Entrega 3, os modelos precisam superar o baseline principalmente no recall da classe positiva, sem produzir uma quantidade inviável de falsos positivos.

## 6. Esboço do fluxo da aplicação

1. **Formulário:** o usuário informa idade, sexo, peso, altura, hipertensão, dias por semana de quatro alimentos e se praticou exercício.
2. **Interface:** envia os campos para a API por `POST /predict`. Ela nunca carrega o modelo diretamente.
3. **API:** valida tipos e faixas, aplica o pipeline salvo (`pipeline.joblib`) e responde em JSON. Peso e altura podem vir em branco: o pré-processamento já sabe imputá-los.
4. **Resposta na tela:** um sinal de rastreamento, com o aviso de que não é diagnóstico e de que é preciso procurar avaliação profissional quando houver alerta.

Exemplo ilustrativo (os valores abaixo são inventados, só para mostrar o formato):

```json
POST /predict
{"idade": 45, "sexo": "feminino", "peso_kg": 78, "altura_cm": 165, "hipertensao": "nao",
 "dias_feijao": 7, "dias_frutas": 3, "dias_refrigerante": 1, "dias_doces": 2, "exercicio": "sim"}

200 OK
{"previsao": "sinal de alerta", "probabilidade": 0.64, "modelo": "pipeline-v1",
 "aviso": "apoio à triagem, não é diagnóstico"}
```

## 7. Riscos e limitações

- **Rótulo de rastreamento:** HbA1c ≥ 6,5% em uma única coleta não equivale a diagnóstico clínico confirmado.
- **Desbalanceamento:** a classe HbA1c ≥ 6,5% é minoritária (3,79%), então métricas por classe são indispensáveis.
- **Dados ausentes:** peso e altura autorreferidos têm ausentes. Eles serão tratados no pipeline, sem excluir linhas.
- **Autorrelato:** peso, altura, hábitos e diagnósticos podem conter erro de memória ou de informação.
- **Vazamento:** `Z034` não entra em X. `Q030` só define a população. Perguntas sobre tratamento ou controle do diabetes não entram como entradas.
- **Colunas sensíveis:** o sexo (`Z001`) e a hipertensão (`Q002`) são dados pessoais e de saúde. O formulário não pede nome nem identificação. O sexo fica como entrada demográfica básica, e seu efeito será reavaliado na E3. A resposta sempre traz o aviso de uso responsável.
- **Representatividade e pesos:** a planilha tem a coluna `peso_lab`. Falta verificar como usá-la, e o baseline desta etapa não é ponderado.
- **Amostra de exames:** a planilha tem 8.952 participantes com dados laboratoriais, e não toda a amostra entrevistada da PNS.
- **Generalização:** os dados são de 2013, e mudanças no perfil e nas práticas de saúde desde então podem afetar o desempenho atual.
- **Uso responsável:** protótipo acadêmico, não é ferramenta de diagnóstico e não substitui exames ou avaliação profissional.

## 8. Referências

- **IBGE — PNS 2013, microdados:** https://www.ibge.gov.br/estatisticas/sociais/saude/9160-pesquisa-nacional-de-saude.html?=&t=microdados
- **IBGE — Nota técnica dos exames laboratoriais da PNS:** https://ftp.ibge.gov.br/PNS/2013/Divulgacoes/Outros/Exames/Nota_tecnica_Exames_Laboratoriais_PNS.pdf
- **Szwarcwald CL, Malta DC, et al.** Exames laboratoriais da Pesquisa Nacional de Saúde: metodologia de amostragem, coleta e análise dos dados. *Rev Bras Epidemiol*, 2019; 22 (Supl 2): e190004.
- **American Diabetes Association — Standards of Care in Diabetes 2026, seção 2:** https://doi.org/10.2337/dc26-S002. A referência considera HbA1c ≥ 6,5% como um critério laboratorial, mas recomenda confirmação diagnóstica na ausência de hiperglicemia inequívoca.
- **Arquivos do projeto:** `exames-PNS-2013-final.xlsx` e `dicionario-variaveis-exames-pns-2013.xlsx`, na pasta `data/` do repositório.